# Notebook 3b: One benchmark for all methods (Setting B, repeated grouped splits)

**What this notebook is for.** Every method we test (source-only, moment matching, AdaBN, DANN, Deep CORAL,
JMMD, JDOT, DeepJDOT, ...) is scored in **exactly the same way**, on the **same 10 grouped calibration/test splits**
of Setting B. The result is one comparison table.

This follows the evaluation protocol of Xiang et al. (2026), so our numbers can be compared with their Table 10.

### The seven metrics
| Metric | Meaning | Better |
|---|---|---|
| **Accuracy** | share of test images classified correctly | higher |
| **ECE (calibrated)** | gap between confidence and accuracy, after temperature scaling | lower |
| **Brier (calibrated)** | squared error of the probabilities, after temperature scaling | lower |
| **AUROC (unknown)** | can the model's confidence tell *known* from *unknown* diseases? 0.5 = coin flip, 1 = perfect | higher |
| **AUPR (unknown)** | the same question, scored by precision and recall with "unknown" as the positive class | higher |
| **Risk@80** | error rate among the answered images when the model aims to answer 80% of cases | lower |
| **Risk@50** | the same, aiming to answer 50% of cases | lower |

### How each split is scored
For every one of the 10 splits:
1. Fit a temperature **T** on that split's **calibration** images.
2. Build **confidence scores** from the temperature-scaled outputs (four kinds, explained in Step 3).
3. For Risk@80 and Risk@50, choose the **threshold on the calibration images** (the confidence level that would answer
   80%, or 50%, of *them*), then **apply that same threshold to the test images**. The share of test images actually
   answered can differ a little from the target. That is realistic: a deployed system cannot peek at its test set.
4. Judge everything on the **test** images (and, for AUROC/AUPR, the unknown pool). Calibration images are only used
   to fit T and to set thresholds.

Then we report the mean and standard deviation over the 10 splits.

### Which confidence score? (as in the paper)
Accuracy, ECE and Brier do not depend on the score. AUROC, AUPR and the risks do. The paper reports, **for each method,
the best-performing of four scores**. We follow that: for every method we pick the score with the highest mean AUROC
across the 10 splits. We also show the table with **MSP only** for every method, as a like-for-like check. (Picking the
best score by looking at results is a mild form of selection, so treat the "best score" table as optimistic.)

### Adding a new method later
Run this notebook again with the new method's name in `METHODS`. Each method's results are saved to their own
file, and the comparison table is rebuilt from **all** saved files, so you never have to recompute old methods.

### Plan
1. Settings
2. Load the split information
3. The metrics (the new ones explained)
4. Score the methods
5. The comparison table
6. Paired differences against the baseline

## Step 1: Settings

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import logsumexp
from sklearn.metrics import roc_auc_score, average_precision_score

In [ ]:
# ---- Edit these -------------------------------------------------------------
METHODS = ["source_only","linear_head", "moment_matching", "adabn", "dann"]   # methods to (re)score now; add new ones here as we build them
REFERENCE = "source_only"                       # the baseline that other methods are compared against
TRAIN_SEED = 42                                  # training seed of the saved outputs

OUT_DIR = Path("outputs")
MANIFEST = OUT_DIR / "master_manifest.csv"
SEEDS = list(range(42, 52))                     # the 10 grouped splits from notebook 01
N_BINS = 15                                     # bins for ECE
# -----------------------------------------------------------------------------

## Step 2: Load the split information

The manifest from notebook 01 says which PlantDoc images are calibration (`cal`) and which are test for each split.
We match the saved outputs to it by image path.

In [12]:
manifest = pd.read_csv(MANIFEST, keep_default_na=False)
split_columns = [f"B_seed{s}" for s in SEEDS]
known_rows = manifest[(manifest["dataset"] == "pd") & (manifest["role"] == "pd_known")].set_index("path")


def load_method(method):
    # Read one method's saved outputs and return what we need to score it
    file = np.load(OUT_DIR / f"outputs_{method}_seed{TRAIN_SEED}.npz", allow_pickle=True)

    def logits(group, seed):
        # Adapted methods save separate logits per split; source-only saves one set for all splits
        per_split_key, shared_key = f"{group}_logits_B_seed{seed}", f"{group}_logits"
        if per_split_key in file.files:
            return file[per_split_key]
        assert shared_key in file.files, f"{method}: no logits found for {group}"
        return file[shared_key]

    return {"logits": logits, "labels": file["pd_known_labels"], "paths": file["pd_known_paths"]}

In [13]:
# Which known PlantDoc image is calibration / test in each split (taken from the first method's image order)
first = load_method(METHODS[0])
split_info = known_rows.loc[first["paths"], split_columns].reset_index(drop=True)
print(len(split_info), "known PlantDoc images")
print("Split 42:", split_info["B_seed42"].value_counts().to_dict())

2018 known PlantDoc images
Split 42: {'test': 1816, 'cal': 202}


## Step 3: The metrics

We start from **probabilities** (softmax of the logits). For each image, **confidence** is the highest probability and
**correct** is 1 if the top class is right.

In [14]:
def softmax(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)


def confidence_and_correct(probs, labels):
    return probs.max(axis=1), (probs.argmax(axis=1) == labels).astype(float)


def ece(confidence, correct, n_bins=N_BINS):
    # Weighted average, over confidence bins, of |average confidence - accuracy|
    edges = np.linspace(0, 1, n_bins + 1)
    total = 0.0
    for low, high in zip(edges[:-1], edges[1:]):
        in_bin = (confidence > low) & (confidence <= high)
        if in_bin.any():
            total += in_bin.mean() * abs(confidence[in_bin].mean() - correct[in_bin].mean())
    return float(total)


def brier(probs, labels):
    one_hot = np.eye(probs.shape[1])[labels]
    return float(((probs - one_hot) ** 2).sum(axis=1).mean())


def nll(probs, labels):
    return float(-np.log(np.clip(probs[np.arange(len(labels)), labels], 1e-12, 1)).mean())


def fit_temperature(logits, labels):
    # The temperature T that minimises the NLL on the calibration images (search over log T)
    loss = lambda log_t: nll(softmax(logits / np.exp(log_t)), labels)
    return float(np.exp(minimize_scalar(loss, bounds=(-3, 3), method="bounded").x))

### Four ways to measure confidence

All four are computed from the **temperature-scaled** logits, and for all four *bigger means more confident*:

* **MSP**: the highest probability.
* **Entropy**: 1 minus the normalised entropy of the probabilities (1 = certain, 0 = completely undecided).
* **Margin**: highest probability minus second highest (how clearly the winner beats the runner-up).
* **Energy**: the log-sum-exp of the logits (the "negative energy" of Liu et al., 2020). It also looks at the size of
  the raw scores, not only their proportions.

In [15]:
SCORES = ["MSP", "Entropy", "Margin", "Energy"]


def confidence_scores(calibrated_logits):
    probs = softmax(calibrated_logits)
    sorted_probs = np.sort(probs, axis=1)
    entropy = -(probs * np.log(np.clip(probs, 1e-12, 1))).sum(axis=1)
    return {
        "MSP": probs.max(axis=1),
        "Entropy": 1 - entropy / np.log(probs.shape[1]),
        "Margin": sorted_probs[:, -1] - sorted_probs[:, -2],
        "Energy": logsumexp(calibrated_logits, axis=1),
    }

### Risk with a transferred threshold

*Selective prediction* means the model answers only when its confidence is above a **threshold**, and otherwise passes
the case to a human. To aim at answering 80% of cases, we look at the **calibration** images and pick the confidence
level that 80% of *them* reach. Then we apply that fixed threshold to the **test** images and measure the error rate
among the answered ones (the **risk**). We also record the share actually answered (the **coverage**), because it may
not be exactly 80%.

In [16]:
def risk_at_coverage(cal_scores, test_scores, test_correct, coverage):
    # Threshold = the score of the n-th most confident calibration image, where n gives the target coverage
    n_keep = int(np.ceil(coverage * len(cal_scores)))
    threshold = np.sort(cal_scores)[::-1][n_keep - 1]

    answered = test_scores >= threshold
    if answered.sum() == 0:
        return np.nan, 0.0                          # the model would refuse everything
    return float(1 - test_correct[answered].mean()), float(answered.mean())

### The new metrics: telling known from unknown diseases

PlantDoc has diseases that PlantVillage never showed the model (the **unknown pool**). A trustworthy model should be
*less confident* on those than on diseases it knows. We test that directly:

* Take the test images of the known classes (**negatives**) and all unknown-pool images (**positives**).
* Give every image a score: **minus its confidence**, so low confidence means "looks unknown".
* **AUROC**: if you pick one random unknown and one random known image, how often does the unknown one get the
  higher score? 0.5 is chance, 1.0 is perfect.
* **AUPR**: sweep a threshold and measure precision and recall for finding the unknowns. Unlike AUROC it depends
  on how common unknowns are (here roughly one image in five), so its "chance level" equals that share.

We do this once for each of the four confidence scores.

In [17]:
def unknown_detection(known_scores, unknown_scores):
    is_unknown = np.r_[np.zeros(len(known_scores)), np.ones(len(unknown_scores))]   # 1 = unknown (positive class)
    looks_unknown = -np.r_[known_scores, unknown_scores]                             # lower confidence = more unknown
    return roc_auc_score(is_unknown, looks_unknown), average_precision_score(is_unknown, looks_unknown)

### Scoring one split

This function is the heart of the notebook. It takes one method's logits for one split and returns every metric we need:
accuracy, calibrated ECE and Brier, and, **for each of the four scores**, the AUROC, AUPR, and the risk and coverage
at 80% and 50%.

In [18]:
def score_split(known_logits, unknown_logits, labels, is_cal):
    is_test = ~is_cal

    # 1. temperature from the calibration images (their labels are used only here)
    T = fit_temperature(known_logits[is_cal], labels[is_cal])

    # 2. calibrated logits for calibration images, test images and the unknown pool
    cal_logits = known_logits[is_cal] / T
    test_logits = known_logits[is_test] / T
    unknown_cal_logits = unknown_logits / T
    test_labels = labels[is_test]

    probs = softmax(test_logits)
    correct = (probs.argmax(axis=1) == test_labels).astype(float)

    row = {
        "Accuracy": correct.mean(),
        "ECE_cal": ece(probs.max(axis=1), correct),
        "Brier_cal": brier(probs, test_labels),
        "T": T, "n_cal": int(is_cal.sum()), "n_test": int(is_test.sum()),
    }

    # 3. everything that depends on the confidence score, once per score
    cal_scores = confidence_scores(cal_logits)
    test_scores = confidence_scores(test_logits)
    unknown_scores = confidence_scores(unknown_cal_logits)

    for name in SCORES:
        auroc, aupr = unknown_detection(test_scores[name], unknown_scores[name])
        row[f"AUROC_{name}"], row[f"AUPR_{name}"] = auroc, aupr
        for coverage in (0.80, 0.50):
            risk, achieved = risk_at_coverage(cal_scores[name], test_scores[name], correct, coverage)
            row[f"Risk@{int(coverage * 100)}_{name}"] = risk
            row[f"Cov@{int(coverage * 100)}_{name}"] = achieved
    return row

## Step 4: Score the methods

For each method in `METHODS`, score all 10 splits and save the table to its own file.

In [19]:
def score_method(method):
    data = load_method(method)
    assert (data["paths"] == first["paths"]).all(), f"{method}: images are in a different order from {METHODS[0]}"

    rows = []
    for seed in SEEDS:
        is_cal = (split_info[f"B_seed{seed}"] == "cal").to_numpy()
        row = score_split(data["logits"]("pd_known", seed), data["logits"]("pd_unknown", seed), data["labels"], is_cal)
        row.update({"method": method, "train_seed": TRAIN_SEED, "split_seed": seed})
        rows.append(row)
    return pd.DataFrame(rows)


for method in METHODS:
    table = score_method(method)
    table.to_csv(OUT_DIR / f"setting_B_metrics_{method}_seed{TRAIN_SEED}.csv", index=False)
    print(f"{method:18s} scored on {len(table)} splits")

source_only        scored on 10 splits
linear_head        scored on 10 splits
moment_matching    scored on 10 splits


In [20]:
# Look at one method, split by split. (The saved file has more columns: every metric for all four scores.)
preview = pd.read_csv(OUT_DIR / f"setting_B_metrics_{METHODS[0]}_seed{TRAIN_SEED}.csv")
preview[["split_seed", "Accuracy", "ECE_cal", "Brier_cal", "T", "AUROC_MSP", "AUPR_MSP",
         "Risk@80_MSP", "Cov@80_MSP", "Risk@50_MSP", "Cov@50_MSP"]].round(4)

,split_seed,Accuracy,ECE_cal,Brier_cal,T,AUROC_MSP,AUPR_MSP,Risk@80_MSP,Cov@80_MSP,Risk@50_MSP,Cov@50_MSP
0,42,0.2390,0.0462,0.8905,5.0218,0.6451,0.3040,0.7360,0.7362,0.7097,0.4989
1,43,0.2384,0.0443,0.8900,5.0058,0.6474,0.3061,0.7459,0.8128,0.7086,0.5215
2,44,0.2357,0.0499,0.8917,4.9800,0.6477,0.3062,0.7481,0.8111,0.7174,0.5105
3,45,0.2395,0.0339,0.8882,5.1836,0.6447,0.3063,0.7427,0.7896,0.6925,0.4334
4,46,0.2347,0.0541,0.8932,4.7793,0.6460,0.3074,0.7458,0.7890,0.7143,0.4937
5,47,0.2379,0.0286,0.8877,5.4357,0.6427,0.3031,0.7434,0.7726,0.6938,0.4532
6,48,0.2347,0.0480,0.8911,4.9829,0.6456,0.3038,0.7497,0.8011,0.7138,0.5025
7,49,0.2373,0.0521,0.8927,4.8047,0.6438,0.3045,0.7443,0.7924,0.6923,0.4438
8,50,0.2368,0.0501,0.8907,4.8520,0.6480,0.3144,0.7473,0.8238,0.7083,0.4983
9,51,0.2423,0.0324,0.8867,5.4160,0.6486,0.3055,0.7344,0.7671,0.7120,0.5584


## Step 5: The comparison table

We read **every** saved method file (not only the ones just scored), so the table grows as we add methods.
Each cell is **mean ± standard deviation over the 10 splits**.

First we reduce each method's wide table to the seven metrics, using a chosen confidence score for the AUROC, AUPR and
risks: either the **best score per method** (highest mean AUROC), or a fixed one such as MSP.

In [21]:
files = sorted(OUT_DIR.glob(f"setting_B_metrics_*_seed{TRAIN_SEED}.csv"))
results = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# Sanity checks: every method has all splits, and every method was judged on the same images
assert results.groupby("method")["split_seed"].nunique().eq(len(SEEDS)).all(), "A method is missing splits"
assert results.groupby("split_seed")["n_test"].nunique().eq(1).all(), "Methods were scored on different test sets"

METRICS = ["Accuracy", "ECE_cal", "Brier_cal", "AUROC_unknown", "AUPR_unknown", "Risk@80", "Risk@50"]
ARROWS = {"Accuracy": "↑", "ECE_cal": "↓", "Brier_cal": "↓", "AUROC_unknown": "↑", "AUPR_unknown": "↑", "Risk@80": "↓", "Risk@50": "↓"}

preferred_order = ["source_only", "linear_head", "adabn", "moment_matching", "dann", "deep_coral", "jmmd", "msun", "jdot", "deepjdot"]
present = list(results["method"].unique())
method_order = [m for m in preferred_order if m in present] + sorted(m for m in present if m not in preferred_order)
print("Methods in the table:", method_order)

Methods in the table: ['source_only', 'linear_head', 'moment_matching']


In [22]:
def per_split_table(results, rule):
    # rule = "best" (best score per method by mean AUROC) or a fixed score name such as "MSP"
    parts = []
    for method, table in results.groupby("method"):
        score = max(SCORES, key=lambda s: table[f"AUROC_{s}"].mean()) if rule == "best" else rule
        parts.append(pd.DataFrame({
            "method": method, "split_seed": table["split_seed"].values, "score": score,
            "Accuracy": table["Accuracy"].values, "ECE_cal": table["ECE_cal"].values, "Brier_cal": table["Brier_cal"].values,
            "AUROC_unknown": table[f"AUROC_{score}"].values, "AUPR_unknown": table[f"AUPR_{score}"].values,
            "Risk@80": table[f"Risk@80_{score}"].values, "Risk@50": table[f"Risk@50_{score}"].values,
            "Cov@80": table[f"Cov@80_{score}"].values, "Cov@50": table[f"Cov@50_{score}"].values,
        }))
    return pd.concat(parts, ignore_index=True)


def comparison_table(frame):
    mean = frame.groupby("method")[METRICS].mean().loc[method_order]
    std = frame.groupby("method")[METRICS].std().loc[method_order]
    table = mean.round(4).astype(str) + " ± " + std.round(4).astype(str)
    table.columns = [f"{m} {ARROWS[m]}" for m in METRICS]
    table.insert(0, "score used", frame.groupby("method")["score"].first().loc[method_order])
    return table


best = per_split_table(results, "best")
msp_only = per_split_table(results, "MSP")

### Table A: best confidence score per method (the paper's convention)

In [23]:
comparison_table(best)

,score used,Accuracy ↑,ECE_cal ↓,Brier_cal ↓,AUROC_unknown ↑,AUPR_unknown ↑,Risk@80 ↓,Risk@50 ↓
method,,,,,,,,
source_only,Entropy,0.2376 ± 0.0023,0.044 ± 0.009,0.8902 ± 0.0021,0.6516 ± 0.0018,0.3078 ± 0.0031,0.7427 ± 0.0045,0.6999 ± 0.0052
linear_head,Entropy,0.227 ± 0.0029,0.0603 ± 0.0052,0.9006 ± 0.0026,0.6252 ± 0.0018,0.2871 ± 0.0025,0.7519 ± 0.0035,0.7265 ± 0.007
moment_matching,MSP,0.2833 ± 0.0051,0.0653 ± 0.0079,0.8699 ± 0.0027,0.6023 ± 0.0067,0.259 ± 0.0052,0.686 ± 0.0063,0.6434 ± 0.0084


### Table B: MSP for every method (like-for-like)

In [24]:
comparison_table(msp_only)

,score used,Accuracy ↑,ECE_cal ↓,Brier_cal ↓,AUROC_unknown ↑,AUPR_unknown ↑,Risk@80 ↓,Risk@50 ↓
method,,,,,,,,
source_only,MSP,0.2376 ± 0.0023,0.044 ± 0.009,0.8902 ± 0.0021,0.646 ± 0.002,0.3061 ± 0.0032,0.7438 ± 0.005,0.7063 ± 0.0097
linear_head,MSP,0.227 ± 0.0029,0.0603 ± 0.0052,0.9006 ± 0.0026,0.6168 ± 0.0019,0.277 ± 0.0024,0.7529 ± 0.0039,0.7379 ± 0.0072
moment_matching,MSP,0.2833 ± 0.0051,0.0653 ± 0.0079,0.8699 ± 0.0027,0.6023 ± 0.0067,0.259 ± 0.0052,0.686 ± 0.0063,0.6434 ± 0.0084


### How much of the target coverage did the threshold really deliver?

We *aimed* to answer 80% and 50% of cases. Because the threshold came from the calibration images, the test images
can be answered more or less often. The paper found about 0.81 at the 80% target for the source-only model.

In [25]:
coverage = best.groupby("method")[["Cov@80", "Cov@50"]].mean().loc[method_order].round(3)
coverage.columns = ["actual coverage at 80% target", "actual coverage at 50% target"]
coverage

,actual coverage at 80% target,actual coverage at 50% target
method,,
source_only,0.798,0.488
linear_head,0.787,0.483
moment_matching,0.808,0.492


**Reading the tables:**
* Arrows show the better direction. A method "wins" a column only if its difference from the others is larger than the
  split-to-split spread (the ± part), so don't call a winner from the means alone.
* The metrics measure *different* things. A method can improve accuracy and leave the model just as overconfident, or
  improve calibration (ECE) while accuracy stays put. That is why we report all seven.
* Compare Table A with Table B: if a method only looks good when the best score is picked, the gain is fragile.
* Compare with the paper's Table 10 (MSP row for the source-only model: accuracy 0.321, ECE 0.365, Brier 1.005,
  AUROC 0.583, AUPR 0.247, Risk@80 0.643, Risk@50 0.594).

## Step 6: Paired differences against the baseline

All methods are judged on the *same* 10 splits, so we can compare them **split by split**: for each split, subtract the
baseline's score from the method's score. Averaging those differences is more informative than comparing two means,
because the "easy" and "hard" splits cancel out. We also count in how many of the 10 splits the method did better.
(A formal significance test comes later. This table is just to build intuition.) We use Table B (MSP for everyone) so the
comparison is not muddied by different methods using different scores.

In [26]:
baseline = msp_only[msp_only["method"] == REFERENCE].set_index("split_seed")[METRICS]

better_is_higher = {"Accuracy": True, "AUROC_unknown": True, "AUPR_unknown": True,
                    "ECE_cal": False, "Brier_cal": False, "Risk@80": False, "Risk@50": False}

rows = []
for method in method_order:
    if method == REFERENCE:
        continue
    other = msp_only[msp_only["method"] == method].set_index("split_seed")[METRICS]
    diff = other - baseline                                    # paired: same split, same row
    for metric in METRICS:
        wins = int(((diff[metric] > 0) if better_is_higher[metric] else (diff[metric] < 0)).sum())
        rows.append({"method": method, "metric": metric,
                     "mean difference": diff[metric].mean(), "std": diff[metric].std(),
                     "splits better (of 10)": wins})

paired = pd.DataFrame(rows)
paired.pivot(index="method", columns="metric", values="mean difference")[METRICS].round(4)

metric,Accuracy,ECE_cal,Brier_cal,AUROC_unknown,AUPR_unknown,Risk@80,Risk@50
method,,,,,,,
linear_head,-0.0107,0.0164,0.0104,-0.0292,-0.0291,0.0091,0.0316
moment_matching,0.0457,0.0213,-0.0203,-0.0437,-0.0472,-0.0578,-0.0629


In [27]:
# In how many of the 10 splits was each method better than the baseline?
paired.pivot(index="method", columns="metric", values="splits better (of 10)")[METRICS]

metric,Accuracy,ECE_cal,Brier_cal,AUROC_unknown,AUPR_unknown,Risk@80,Risk@50
method,,,,,,,
linear_head,0,0,0,0,0,0,0
moment_matching,10,0,10,0,0,10,10


**Questions to think about:**
* Is any method *consistently* better (say 9 or 10 splits out of 10) on a metric, or only better on average?
* Do methods that improve accuracy also improve ECE and the risks, or do the metrics disagree?
* Is AUROC (unknown) near 0.5? What would that tell you about the model's confidence on diseases it has never seen?

### Next
Each new method (AdaBN, DANN, Deep CORAL, JMMD, JDOT, DeepJDOT) saves its outputs in the same format, gets added to
`METHODS`, and appears as a new row. After that we add a formal significance test (paired Wilcoxon with Holm
correction), and repeat over several training seeds.